# Animal Biological Class Predictor: A Beginner-Friendly Machine Learning Walkthrough

This notebook provides an interactive walkthrough of a complete **supervised machine learning multiclass classification** pipeline.

### The Pipeline:
```text
Data Acquisition -> Inspection -> Preprocessing -> Stratified Split -> Model Training -> Validation & Tuning -> Test Evaluation -> Prediction
```

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import joblib

print('Libraries loaded successfully!')

## 1. Inspecting the Dataset
We load the raw UCI Zoo dataset and filter to the 5 vertebrate classes: **Mammal, Bird, Reptile, Fish, Amphibian**.

In [ ]:
columns = ['animal_name', 'hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail', 'domestic', 'catsize', 'class_type']
class_map = {1: 'Mammal', 2: 'Bird', 3: 'Reptile', 4: 'Fish', 5: 'Amphibian'}
df_raw = pd.read_csv('../data/raw/zoo.data', header=None, names=columns)
df = df_raw[df_raw['class_type'].isin(class_map.keys())].copy()
df['class_name'] = df['class_type'].map(class_map)
print(f'Dataset Shape: {df.shape}')
print(df['class_name'].value_counts())

## 2. Train / Validation / Test Splits
We load the pre-split, stratified datasets created by `src/preprocess.py` to guarantee zero data leakage.

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

features = [c for c in train_df.columns if c not in ['animal_name', 'class_type', 'class_name']]
X_train, y_train = train_df[features], train_df['class_name']
X_val, y_val = val_df[features], val_df['class_name']
X_test, y_test = test_df[features], test_df['class_name']

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 3. Baseline Model vs. Decision Tree
A baseline predicts the most frequent class (Mammal). We compare this against our Decision Tree.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print(f'Baseline Validation Accuracy: {accuracy_score(y_val, baseline.predict(X_val)):.4f}')

dt = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt.fit(X_train, y_train)
print(f'Decision Tree Validation Accuracy: {accuracy_score(y_val, dt.predict(X_val)):.4f}')

## 4. Visualizing the Decision Tree
Decision Trees are white-box models. We can inspect every decision rule learned from data.

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(dt, feature_names=features, class_names=list(dt.classes_), filled=True, rounded=True, fontsize=8)
plt.title('Learned Decision Tree', fontsize=14, fontweight='bold')
plt.show()

## 5. Final Evaluation on Untouched Test Set
We evaluate only ONCE on the test set.

In [ ]:
y_pred = dt.predict(X_test)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=dt.classes_)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('Test Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 6. Live Prediction on a Mystery Animal
Input a novel animal attribute vector to get predicted class and probabilities.

In [ ]:
mystery_animal = {
    'hair': 0, 'feathers': 1, 'eggs': 1, 'milk': 0, 'airborne': 1,
    'aquatic': 0, 'predator': 0, 'toothed': 0, 'backbone': 1, 'breathes': 1,
    'venomous': 0, 'fins': 0, 'legs': 2, 'tail': 1, 'domestic': 0, 'catsize': 0
}
mystery_df = pd.DataFrame([[mystery_animal[f] for f in features]], columns=features)
pred = dt.predict(mystery_df)[0]
probs = dt.predict_proba(mystery_df)[0]
print(f'Predicted Class: {pred}')
for cls, p in zip(dt.classes_, probs):
    print(f'  {cls:<10}: {p:.2%}')